<a href="https://colab.research.google.com/github/PatrickCalorioCarvalho/CA015ICMineracaoDeDados202601/blob/main/Pr%C3%A9_processamento_de_Dados_com_Scikit_Learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Análise Inicial dos Dados
Realize uma análise exploratória básica contendo:

* Quantidade de registros e atributos;
* Tipos de dados presentes na base;
* Quantidade de valores ausentes por atributo;
* Distribuição da variável alvo Churn.

In [18]:
#Quantidade de registros e atributos
linhas, colunas = df.shape
print(f"Registros: {linhas}")
print(f"Atributos: {colunas}")

Registros: 7043
Atributos: 21


In [45]:
#Tipos de dados presentes na base
for coluna in df.columns:
    print(f"{coluna} : ", df[coluna].dtype)

customerID :  object
gender :  object
SeniorCitizen :  float64
Partner :  object
Dependents :  object
tenure :  float64
PhoneService :  object
MultipleLines :  object
InternetService :  object
OnlineSecurity :  object
OnlineBackup :  object
DeviceProtection :  object
TechSupport :  object
StreamingTV :  object
StreamingMovies :  object
Contract :  object
PaperlessBilling :  object
PaymentMethod :  object
MonthlyCharges :  float64
TotalCharges :  object
Churn :  object


In [39]:
#Quantidade de valores ausentes por atributo
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [33]:
#Distribuição da variável alvo Churn
print(df["Churn"].value_counts())

Churn
No     5174
Yes    1869
Name: count, dtype: int64


# Pré-processamento dos Dados
Utilizando a biblioteca Scikit-Learn, implemente:

* Tratamento de valores ausentes: utilizar SimpleImputer para tratar atributos com valores faltantes.
* Transformação de atributos categóricos: utilizar OneHotEncoder para atributos categóricos.
* Normalização: utilizar MinMaxScaler ou StandardScaler para atributos numéricos.
* Remoção de atributos irrelevantes: justificar a remoção de atributos que não contribuem para a predição, como identificadores, por exemplo.

In [69]:
#Remoção de atributos irrelevantes: justificar a remoção de atributos que não contribuem para a predição, como identificadores, por exemplo
df = df.drop(columns=["customerID"]) # remover Coluna de Indeificacao Unica


X = df.drop(columns=["Churn"]) # remover class
y = df["Churn"] # separa class previsao


In [70]:
#Tratamento de valores ausentes: utilizar SimpleImputer para tratar atributos com valores faltantes.

numeric_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist() #pegar todas as colunas numericas
imputer_num = SimpleImputer(strategy="mean")
X[numeric_cols] = imputer_num.fit_transform(X[numeric_cols]) #aplicando as valores faltante as media



In [71]:
#Transformação de atributos categóricos: utilizar OneHotEncoder para atributos categóricos.

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

encoder = OneHotEncoder(sparse_output=False)
# Retorna em formato array
Dado_encoder = encoder.fit_transform(X[categorical_cols])
# Seleciona o nome das colunas geradas
Coluna_names = encoder.get_feature_names_out(categorical_cols)
# Transformar em um Dataframe normalizado
ndf = pd.DataFrame(
    Dado_encoder,
    columns = Coluna_names,
    index= df.index
)
X = X.drop(columns=categorical_cols)
X = pd.concat([X, ndf], axis=1)

In [72]:
#Normalização: utilizar MinMaxScaler ou StandardScaler para atributos numéricos.
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])


# Separação dos Dados
Realize a divisão dos dados em:

* Treinamento: 80%
* Teste: 20%

Utilize a função:

> train_test_split()


In [73]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state= 2,
    stratify = y
)

# Evitando Data Leakage
O pré-processamento deve ser realizado corretamente para evitar vazamento de dados.

Portanto:

* Os transformadores SimpleImputer, OneHotEncoder e Scaler devem ser ajustados com fit apenas no conjunto de treinamento;
* O conjunto de teste deve utilizar apenas a transformação com transform, gerada a partir do conjunto de treinamento;
* Explique brevemente no relatório o que é Data Leakage e por que ele deve ser evitado.

# Construção do Modelo
Treine um modelo utilizando:

> DecisionTreeClassifier

Avalie o modelo utilizando:
*  Relatório de Classificação, contendo Precision, Recall e F1-Score.

In [79]:
#Treine um modelo utilizando
modelo = DecisionTreeClassifier()

In [80]:
#Os transformadores SimpleImputer, OneHotEncoder e Scaler devem ser ajustados com fit apenas no conjunto de treinamento;
modelo.fit(X_train, y_train)

DecisionTreeClassifier()

In [81]:
#O conjunto de teste deve utilizar apenas a transformação com transform, gerada a partir do conjunto de treinamento;
y_pred = modelo.predict(X_test)

Explique brevemente no relatório o que é Data Leakage e por que ele deve ser evitado.

acontece quando informação do conjunto de teste — ou
qualquer informação que não estaria disponível no momento real da previsão — "escapa" para o
processo de treinamento do modelo


In [83]:
#Relatório de Classificação, contendo Precision, Recall e F1-Score.
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.84      0.86      0.85      1035
         Yes       0.58      0.53      0.55       374

    accuracy                           0.77      1409
   macro avg       0.71      0.70      0.70      1409
weighted avg       0.77      0.77      0.77      1409

